In [7]:
import os
print("Lokasi notebook saat ini:", os.getcwd())
print("Isi folder ini:", os.listdir())

Lokasi notebook saat ini: /workspaces/cardioguard-project/ai-engineering/scripts
Isi folder ini: ['.gitkeep', 'training_model.py', 'data_preparation.ipynb']


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pickle

df = pd.read_csv('../../data/processed/cardio_cleaned.csv')

In [9]:
df.head()

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,age_years,bmi,cardio
0,2,168,62.0,110,80,1,1,0,0,1,50,21.97,0
1,1,156,85.0,140,90,3,1,0,0,1,55,34.93,1
2,1,165,64.0,130,70,3,1,0,0,0,52,23.51,1
3,2,169,82.0,150,100,1,1,0,0,1,48,28.71,1
4,1,156,56.0,100,60,1,1,0,0,0,48,23.01,0


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 68540 entries, 0 to 68539
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   gender       68540 non-null  int64  
 1   height       68540 non-null  int64  
 2   weight       68540 non-null  float64
 3   ap_hi        68540 non-null  int64  
 4   ap_lo        68540 non-null  int64  
 5   cholesterol  68540 non-null  int64  
 6   gluc         68540 non-null  int64  
 7   smoke        68540 non-null  int64  
 8   alco         68540 non-null  int64  
 9   active       68540 non-null  int64  
 10  age_years    68540 non-null  int64  
 11  bmi          68540 non-null  float64
 12  cardio       68540 non-null  int64  
dtypes: float64(2), int64(11)
memory usage: 6.8 MB


## Feature Engineering

Untuk meningkatkan daya pelajari model, dilakukan feature engineering dengan membentuk variabel kategori tekanan darah (bp_cat). Batasan ambang klasifikasi yang digunakan merujuk pada pedoman klinis resmi dari American Heart Association dan American College of Cardiology tahun 2017 (Whelton et al., 2018), yang membaginya ke dalam kategori Normal, Elevated, Hipertensi Stage 1, dan Hipertensi Stage 2

In [11]:
# kategori Tekanan Darah (Ordinal)
def bp_category(row):
    hi = row['ap_hi']
    lo = row['ap_lo']
    if hi < 120 and lo < 80:
        return 0 # Normal
    elif hi < 130 and lo < 80:
        return 1 # Elevated
    elif hi < 140 or lo < 90:
        return 2 # Hipertensi Stage 1
    else:
        return 3 # Hipertensi Stage 2

df['bp_cat'] = df.apply(bp_category, axis=1)

selain itu, kita juga mengekstraksi fitur baru, pulse pressure. Fitur ini secara matematis dikalkulasi melalui selisih antara nilai tekanan darah sistolik dan diastolik. Secara klinis, representasi nilai tekanan nadi merupakan indikator penting dalam mengukur tingkat arterial stiffness dan penuaan pembuluh darah, yang diakui secara luas dalam literatur kardiologi sebagai prediktor independen yang kuat terhadap peningkatan risiko kejadian kardiovaskular

In [12]:
# pulse pressure / tekanan nadi 
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']

Dibuat juga fitur Mean Arterial Pressure (MAP), yang diestimasi menggunakan formula klinis standar, yakni tekanan diastolik ditambah sepertiga tekanan nadi. Dalam literatur kardiovaskular, parameter ini diakui secara luas sebagai indikator hemodinamik yang lebih akurat untuk mengevaluasi perfusi jaringan (kecukupan suplai darah ke organ vital) dibandingkan jika model hanya mengevaluasi tekanan sistolik dan diastolik secara terpisah.

In [13]:
# Mean Arterial Pressure (MAP)
df['map'] = df['ap_lo'] + (df['pulse_pressure'] / 3)

In [14]:
print(f"Feature engineering selesai! Total kolom sekarang: {df.shape[1]}")
display(df[['ap_hi', 'ap_lo', 'pulse_pressure', 'map', 'bp_cat']].head())

Feature engineering selesai! Total kolom sekarang: 16


,ap_hi,ap_lo,pulse_pressure,map,bp_cat
0,110,80,30,90.000000,2
1,140,90,50,106.666667,3
2,130,70,60,90.000000,2
3,150,100,50,116.666667,3
4,100,60,40,73.333333,0


## Data prep for train

In [15]:
# pisah fitur dan target model
X = df.drop('cardio', axis=1)
y = df['cardio']

In [16]:
# scaling fitur
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [18]:
# split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Jumlah data train: {X_train.shape[0]} baris")
print(f"Jumlah data test: {X_test.shape[0]} baris")

Jumlah data train: 54832 baris
Jumlah data test: 13708 baris


In [19]:
# save to folder data ai-engineering
X_train.to_csv('../data/X_train_scaled.csv', index=False)
X_test.to_csv('../data/X_test_scaled.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

In [20]:
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [21]:
print("\nData Siap!")
display(X_train.head())


Data Siap!


,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,age_years,bmi,bp_cat,pulse_pressure,map
17102,1.0,0.506494,0.303030,0.40625,0.4,0.5,0.0,1.0,0.0,1.0,0.457143,0.238134,1.000000,0.370370,0.426471
24074,0.0,0.363636,0.103030,0.12500,0.1,0.0,0.0,0.0,0.0,1.0,0.571429,0.116687,0.000000,0.259259,0.117647
49087,1.0,0.493506,0.200000,0.25000,0.3,1.0,1.0,0.0,0.0,1.0,0.742857,0.161023,0.666667,0.259259,0.294118
30748,1.0,0.584416,0.466667,0.50000,0.6,1.0,0.0,0.0,0.0,1.0,0.685714,0.330749,1.000000,0.333333,0.588235
20789,0.0,0.363636,0.248485,0.25000,0.3,1.0,1.0,0.0,0.0,0.0,0.971429,0.247382,0.666667,0.259259,0.294118
